%md
##### Note: this notebook has two parts.

##### Blocks 1-4 are the actual production flow  parses orders with a flexible MapType (no fixed schema needed) writes via MERGE keyed on topic+partition+offset, so it's safe to re run without creating duplicates. Any new field in the JSON (like discount_code) is picked up automatically no schema changes required on this side.

##### Blocks 5-7 are a separate deliberate  of schema enforcement/evolution using a strict StructType and a plain write  to show what happens when a table's schema doesn't tolerate a new field. This part is not idempotent  re running it appends the same rows again. Run it once to capture the evidence don't fold it into a scheduled job.

In [12]:
%run ../setup/config

Box(children=(Label(value='Environment'), Dropdown(options=('dev', 'prod'), value='dev')))

Box(children=(Label(value='Cluster ID (vacío = serverless)'), Text(value='')))

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import MapType, StringType
from delta.tables import DeltaTable

In [14]:
df_bronze_orders = spark.table(BRONZE_ORDER_FULL_TABLE)

df_parsed = df_bronze_orders.withColumn(
    "event",
    F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))
)

df_silver_orders = (
    df_parsed
    .select(
        F.col("event")["order_id"].cast("int").alias("order_id"),
        F.col("event")["item_id"].cast("int").alias("item_id"),
        F.col("event")["event_timestamp"].cast("timestamp").alias("event_timestamp"),
        F.col("event")["discount_code"].alias("discount_code"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.current_timestamp().alias("_ingestion_timestamp"),
    )
)

silver_orders_table = DeltaTable.forName(spark, SILVER_ORDER_FULL_TABLE)

(
    silver_orders_table.alias("tgt")
    .merge(
        df_silver_orders.alias("src"),
        "tgt.topic = src.topic AND tgt.`partition` = src.`partition` AND tgt.`offset` = src.`offset`"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"MERGE applied for {df_silver_orders.count()} candidate rows (only new offsets inserted)")

MERGE applied for 1010 candidate rows (only new offsets inserted)


%md

##### from here on: a deliberate  of schema enforcement and evolution using a strict schema on purpose. Not part of the regular pipeline, run once for evidence, don't schedule it.

In [ ]:
# isolate the schema_change scenario events (they  discount_code a field ORDER_EVENT_SCHEMA below doesn't declare on purpose for this demo)


df_bronze_schema_change = (
    spark.table(BRONZE_ORDER_FULL_TABLE)
    .filter(F.col("raw_json").contains("discount_code"))
)

df_parsed_v2 = df_bronze_schema_change.withColumn(
    "event",
    F.from_json(F.col("raw_json"), ORDER_EVENT_SCHEMA_V2)
)

df_silver_orders_v2 = (
    df_parsed_v2
    .select(
        F.col("event.order_id").alias("order_id"),
        F.col("event.item_id").alias("item_id"),
        F.col("event.event_timestamp").cast("timestamp").alias("event_timestamp"),
        F.col("event.discount_code").alias("discount_code"),
        F.col("topic"),
        F.col("partition"),
        F.col("offset"),
        F.current_timestamp().alias("_ingestion_timestamp"),
    )
)

In [ ]:
# schema enforcement : plain write without mergeSchema.
# expected to fail with DELTA_METADATA_MISMATCH the first time this table sees
# a column it doesn't have yet. Caught here so the notebook can still complete.


try:
    (
        df_silver_orders_v2.write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_ORDER_FULL_TABLE)
    )
except Exception as e:
    print(f"Expected schema enforcement error:\n{e}")

# schema evolution : retry with mergeSchema=true  delta widens the table
# automatically instead of rejecting the write.

(
    df_silver_orders_v2.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_ORDER_FULL_TABLE)
)

print(f"{df_silver_orders_v2.count()} rows written to {SILVER_ORDER_FULL_TABLE} (with schema evolution)")

In [15]:
duplicate_offsets = (
    spark.table(SILVER_ORDER_FULL_TABLE)
    .groupBy("topic", "partition", "offset")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Duplicate topic+partition+offset combinations: {duplicate_offsets}")

0